# 03 · Customer-level features & RFM analysis
*Sections 8–10.*

## 8 · Customer-level feature engineering
| Feature | Definition | Why |
|---|---|---|
| **Recency (R)** | days from last *purchase* to snapshot (10 Dec 2011) | returns do not count as engagement |
| **Frequency (F)** | unique purchase invoices | invoices, **not lines** |
| **Monetary (M)** | **net** revenue = sales − returns | the guideline's "sum of all transaction revenues" |
| AOV, AvgBasketUnits, UniqueProducts, ReturnRate, Tenure | context for profiling | explain *why* a segment behaves as it does |

Excluded from customer features: guests (no identity), accounting adjustments, zero-price lines, non-product fees.

In [1]:
import sys, warnings; sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, plotly.express as px, plotly.graph_objects as go
from src import cleaning as cl, features as ft, segmentation as sg, viz
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = 40
df = pd.read_parquet("../data/processed/lines_clean.parquet")
cust = ft.customer_table(df)
print(f"{len(cust):,} identified customers | snapshot = {ft.snapshot_date(df).date()}")
print(f"{(cust.Monetary <= 1).sum()} customers have net revenue <= £1 (everything returned) -> cannot be scored")
cust[["Recency","Frequency","Monetary","AOV","AvgBasketUnits","ReturnRate"]].describe(percentiles=[.5,.9,.99]).round(1)

4,334 identified customers | snapshot = 2011-12-10
14 customers have net revenue <= £1 (everything returned) -> cannot be scored


,Recency,Frequency,Monetary,AOV,AvgBasketUnits,ReturnRate
count,"4,334.00","4,334.00","4,334.00","4,334.00","4,334.00","4,334.00"
mean,93.20,4.20,"1,908.00",415.50,255.30,0.00
std,100.20,7.60,"8,291.10","1,800.90","1,316.40",0.10
min,1.00,1.00,-141.50,3.80,1.00,0.00
50%,51.00,2.00,646.80,289.60,161.00,0.00
90%,264.00,9.00,"3,449.40",657.80,421.20,0.00
99%,369.00,30.00,"17,049.10","2,023.80","1,385.00",0.40
max,374.00,206.00,"278,778.00","84,236.20","74,215.00",1.00


Frequency and Monetary are extremely right-skewed (mean ≫ median, max ≫ p99). This is the wholesale effect and is why **we log-transform before clustering**.

## 9 · RFM analysis
**Scoring direction (read this before any chart):** each metric is cut into **5 quintiles**.
* **R: 5 = bought most recently** (lowest days) — the direction is inverted on purpose.
* **F: 5 = most invoices**, **M: 5 = most net revenue**.
* Score `'555'` = best on all three; `'155'` = big and frequent but gone quiet.

Technical note: `pd.qcut` fails on Frequency because ~35% of customers have exactly 1 invoice (tied edges). We use `rank(method="first")` before `qcut` so bins stay equal-sized.

In [2]:
rfm = ft.score_rfm(cust)
print(f"{len(rfm):,} customers scored")
rfm[["CustomerID","Recency","Frequency","Monetary","R","F","M","RFM_Score","RFM_Segment"]].sort_values("Monetary", ascending=False).head(8)

4,320 customers scored


,CustomerID,Recency,Frequency,Monetary,R,F,M,RFM_Score,RFM_Segment
1689,14646,2,72,"278,778.02",5,5,5,555,Champions
4197,18102,1,60,"259,657.30",5,5,5,555,Champions
3725,17450,9,46,"189,575.53",5,5,5,555,Champions
1879,14911,2,198,"128,768.24",5,5,5,555,Champions
55,12415,25,20,"123,638.18",4,5,5,455,Champions
1333,14156,10,54,"113,685.77",5,5,5,555,Champions
3768,17511,3,31,"88,138.20",5,5,5,555,Champions
3174,16684,5,28,"65,920.12",5,5,5,555,Champions


In [3]:
(rfm.groupby("R").Recency.agg(["min","max","count"]).rename_axis("R score").T)

R score,1,2,3,4,5
min,180,72,33,15,1
max,374,180,72,33,13
count,864,864,864,864,864


### Segment rules (derived from the business matrix, not forced)
| Segment | Rule | Strategy |
|---|---|---|
| **Champions** | R≥4, F≥4, M≥4 | loyalty perks, early access, referrals |
| **Loyal Customers** | R≥3, F≥3 (rest) | cross-sell related catalog |
| **At-Risk High-Value** | R≤2, F≥3, M≥3 | immediate win-back |
| **New Inflow** | R≥4, F≤2 | welcome sequence → 2nd purchase |
| **Hibernating / Lost** | R≤2, F≤2 | low-cost reactivation or let go |
| **Needs Attention** | everything else | monitor |

The guideline sketches Champions as exactly `555`; with 5 quintiles that would be a handful of people, so we use `≥4` on each axis. This is a **judgement call and is stated, not hidden**.

In [4]:
seg = rfm.groupby("RFM_Segment").agg(Customers=("CustomerID","count"), Median_R=("Recency","median"), Median_F=("Frequency","median"),
        Median_M=("Monetary","median"), Revenue=("Monetary","sum"))
seg["Cust_%"] = 100*seg.Customers/seg.Customers.sum(); seg["Rev_%"] = 100*seg.Revenue/seg.Revenue.sum()
seg.sort_values("Rev_%", ascending=False).round(1)

,Customers,Median_R,Median_F,Median_M,Revenue,Cust_%,Rev_%
RFM_Segment,,,,,,,
Champions,940,11.00,7.00,"2,588.30","5,524,771.40",21.80,66.80
Loyal Customers,996,34.00,3.00,752.60,"1,252,090.80",23.10,15.10
At-Risk High-Value,459,118.00,3.00,"1,019.40","713,577.00",10.60,8.60
Hibernating / Lost,1072,219.00,1.00,275.80,"430,001.50",24.80,5.20
Needs Attention,544,64.00,1.00,314.50,"217,467.90",12.60,2.60
New Inflow,309,19.00,1.00,307.10,"131,728.80",7.20,1.60


## 10 · RFM visualisation & interpretation

In [5]:
viz.rfm_heatmap(rfm)

In [6]:
viz.value_vs_size(rfm)

In [7]:
viz.segment_treemap(rfm)

**Interpretation**
* **Champions are 22% of customers but 67% of net revenue** (median 7 orders, last bought ~11 days ago). They are the business.
* **At-Risk High-Value: 459 customers, £714k (8.6% of revenue)** that used to order ~3+ times and have been silent for ~4 months (median recency 118 days) — the most valuable *recoverable* money.
* **Hibernating / Lost: the largest segment by headcount (1,072, 25%) but only 5% of revenue** — do not spend premium retention budget here.
* **New Inflow is tiny (309, 1.6% of revenue)**: recent single-order customers. Converting them to a second order is the cheapest growth lever.

The RFM grid heatmap confirms the structure: spend rises steeply with *frequency*, and recency matters mostly in the low-F rows.

In [8]:
rfm.to_parquet("../data/processed/customers_rfm_only.parquet")
rfm.RFM_Score.value_counts().head(10).to_frame("customers")

,customers
RFM_Score,
555,346
111,188
455,174
121,152
112,120
444,114
122,93
344,87
233,87
